# 01. Угроза №28: конденсация на датчиках (роса / запотевание)

**Категория угрозы:** фазовый переход влаги при смене температуры → запотевание объективов, искажение данных тепловизоров и лидаров.
**Последствия для миссии:** потеря качества съёмки, ошибки компьютерного зрения.

## Связь с данными из ТЗ
| Сигнал | Поток ТЗ | Как используем |
|---|---|---|
| `temperature`, `humidity` | БВС-5 | точка росы Td (Magnus), запас T − Td |
| `alt` | БВС-1 | перепад условий при снижении |
| видеопоток | вне потоков ТЗ | резкость кадра (дисперсия лапласиана) |

## Идея
1. **Физический канал (гипотеза: упреждающий).** Температура линзы *не измеряется*, поэтому оцениваем её звеном первого порядка по температуре воздуха и сравниваем с Td. Ожидаем предупреждение *до* появления тумана на стекле; проверяем это в разделах 4-6.
2. **Видеоканал (подтверждающий).** Запотевание проявляется как равномерное падение резкости; ловим его CUSUM по log-резкости.
3. **Слияние:** предупреждение (физика) + подтверждение (видео).

> **Допущение для отчёта:** реальных данных нет, поэтому все результаты получены на синтетических сценариях с известной истиной. Это проверка работоспособности метода, а не оценка точности на борту.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.physics import dewpoint, rel_humidity, wet_bulb_stull
from src.synth_fog import (FogScenario, simulate, make_texture, render_frame, lap_var, sharpness_lut, random_scenario)
from src.fog_detectors import run_pipeline, event_stats, monte_carlo, summarize, physics_detector
plt.rcParams["figure.dpi"] = 110

## 1. Проверка физических формул
Контрольная точка из Stull (2011): T = 20 °C, RH = 50 % → Tw ≈ 13.7 °C; Td по Magnus ≈ 9.3 °C.

In [ ]:
Td = float(dewpoint(20, 50)); Tw = float(wet_bulb_stull(20, 50))
print(f"Td = {Td:.2f} °C, Tw = {Tw:.2f} °C")
assert abs(Td - 9.3) < 0.2 and abs(Tw - 13.7) < 0.2
assert abs(float(rel_humidity(20, Td)) - 50) < 0.5      # обратное преобразование

T = np.linspace(-10, 35, 100)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
for rh in (30, 50, 70, 90):
    ax.plot(T, dewpoint(T, rh), label=f"RH={rh}%")
ax.plot(T, T, "k--", lw=0.8); ax.set_xlabel("T, °C"); ax.set_ylabel("Td, °C"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Точка росы (Magnus)"); plt.show()

## 2. Модель синтетического сценария
Дрон летит на высоте, где воздух холоднее и суше, затем снижается в тёплый влажный приземный слой. Линза инерционна и остаётся холодной дольше воздуха:

- линза: `dT_lens/dt = (T_air + offset − T_lens) / τ` (offset < 0: выстывание/обдув, > 0: нагрев электроникой);
- конденсат: `f' = gain·max(Td − T_lens, 0)·(1−f) − decay·f·[Td ≤ T_lens]`;
- кадр: гауссово размытие σ = 6·f px и потеря контраста до 50 %;
- датчики БВС-5: шум (σT = 0.3 К, σRH = 2 %) и инерция 10 с;
- сцена меняется во времени (AR(1) в лог-шкале): это главный источник ложных срабатываний видеоканала.

Метка «истина»: `f > 0.15`. Детекторы её не видят.

In [ ]:
tex = make_texture(seed=1)
fig, axs = plt.subplots(1, 4, figsize=(11, 3))
for a, f in zip(axs, (0.0, 0.2, 0.5, 1.0)):
    fr = render_frame(tex, f); a.imshow(fr, cmap="gray", vmin=0, vmax=1); a.axis("off")
    a.set_title(f"f={f}\nрезкость={lap_var(fr)/lap_var(tex):.3f}")
plt.show()

In [ ]:
sc = FogScenario(seed=1)
df = simulate(sc)
fig, axs = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
axs[0].plot(df.t, df.alt); axs[0].set_ylabel("высота, м")
axs[1].plot(df.t, df.T_air, label="T воздуха"); axs[1].plot(df.t, df.Td_air, label="Td")
axs[1].plot(df.t, df.T_lens, label="T линзы (скрыта)"); axs[1].legend(ncol=3); axs[1].set_ylabel("°C")
axs[2].plot(df.t, df.fog, c="C3"); axs[2].axhline(0.15, c="k", ls=":"); axs[2].set_ylabel("запотевание f")
axs[3].semilogy(df.t, df.sharp, c="C2"); axs[3].set_ylabel("резкость (отн.)"); axs[3].set_xlabel("t, с")
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. Детекторы на одном сценарии
- `warn_phys`: физический канал (T, RH → Td, оценка T_lens, вероятность по нормальному закону, гистерезис).
- `alarm_video`: CUSUM по падению log-резкости относительно медленной базы.
- `alarm_fused`: срабатывание любого канала; `confirmed`: оба канала.

In [ ]:
res = run_pipeline(df)
print(event_stats(res))
fig, axs = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
axs[0].plot(res.t, res.fog, label="истина f"); axs[0].plot(res.t, res.p_phys, label="p (физика)", alpha=.8)
axs[0].legend(); axs[0].set_ylim(-.05, 1.05)
axs[1].plot(res.t, res.drive, label="Td − T_lens (оценка)"); axs[1].axhline(0, c="k", ls=":"); axs[1].legend()
axs[2].fill_between(res.t, 0, res.warn_phys*1.0, alpha=.4, step="mid", label="warn_phys")
axs[2].fill_between(res.t, 1.1, 1.1+res.alarm_video*1.0, alpha=.4, step="mid", color="C1", label="alarm_video")
axs[2].fill_between(res.t, 2.2, 2.2+res.confirmed*1.0, alpha=.4, step="mid", color="C2", label="confirmed")
axs[2].legend(loc="upper left"); axs[2].set_xlabel("t, с")
onset = res.t[res.label].iloc[0]
for a in axs: a.axvline(onset, c="r", ls="--", lw=.8); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. Статистика на случайных сценариях (Monte-Carlo)
Случайные T, Td, высота, τ линзы (40–300 с), offset (−3…+1 К), скорость снижения. Часть сценариев запотевания не вызывает: на них считаем ложные тревоги.

- **recall:** тревога в окне [onset − 300 с; onset + 120 с];
- **delay:** время первой тревоги минус onset (отрицательное = *раньше* события);
- **false_alarm_rate:** доля сценариев *без* события, где была хоть одна тревога. Для физического канала это включает «почти-запотевания» (запас < 0.5 К), поэтому цифра консервативна.

In [ ]:
mc = monte_carlo(n=300, seed=0)
display(summarize(mc).round(2))
pos = mc[mc.has_event]
fig, ax = plt.subplots(figsize=(6, 3.2))
for ch, lab in [("warn_phys", "физика"), ("alarm_video", "видео"), ("alarm_fused", "слияние")]:
    ax.hist(pos[f"{ch}_delay"].dropna(), bins=np.arange(-300, 125, 15), alpha=.5, label=lab)
ax.axvline(0, c="r", ls="--"); ax.set_xlabel("задержка относительно onset, с"); ax.legend(); plt.show()

## 5. Чувствительность: что если модель линзы неточна?
В реальности τ и offset неизвестны. Смотрим, как «ошибочные» параметры физического канала влияют на recall и ложные тревоги. Истинные τ ∈ [40; 300] с и offset ∈ [−3; +1] К.

In [ ]:
taus, offs = [60, 120, 240], [-2.0, 0.0, 1.0]
R = np.zeros((len(offs), len(taus))); F = np.zeros_like(R)
for i, o in enumerate(offs):
    for j, t in enumerate(taus):
        s = summarize(monte_carlo(n=120, seed=1, tau_hat=t, offset_hat=o)).loc["warn_phys"]
        R[i, j], F[i, j] = s.recall, s.false_alarm_rate
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
for a, M, ttl in zip(axs, (R, F), ("recall (физика)", "false alarm rate (физика)")):
    im = a.imshow(M, vmin=0, vmax=1, cmap="viridis"); a.set_title(ttl)
    a.set_xticks(range(len(taus)), taus); a.set_yticks(range(len(offs)), offs)
    a.set_xlabel("tau_hat, с"); a.set_ylabel("offset_hat, К")
    for (i, j), v in np.ndenumerate(M): a.text(j, i, f"{v:.2f}", ha="center", va="center", c="w")
plt.tight_layout(); plt.show()

## 6. Компромисс «упреждение vs ложные тревоги»
Параметр `margin_K` задаёт, за сколько кельвинов до пересечения Td мы предупреждаем.

In [ ]:
rows = []
for m in (-0.5, 0.0, 0.5, 1.0, 1.5):
    s = summarize(monte_carlo(n=150, seed=2, margin_K=m))
    for ch in ("warn_phys", "alarm_fused"):
        rows.append(dict(margin_K=m, channel=ch, **s.loc[ch, ["recall", "median_delay_s", "false_alarm_rate"]]))
tr = pd.DataFrame(rows)
display(tr.pivot(index="margin_K", columns="channel", values=["recall", "median_delay_s", "false_alarm_rate"]).round(2))
fig, ax = plt.subplots(figsize=(5, 3.4))
for ch, g in tr.groupby("channel"): ax.plot(g.false_alarm_rate, g.recall, "o-", label=ch)
ax.set_xlabel("false alarm rate"); ax.set_ylabel("recall"); ax.legend(); ax.grid(alpha=.3); plt.show()

## 7. Выходной интерфейс модуля
Результат передаётся дальше в Confidence Monitoring / Decision Arbiter и на НСУ. В ТЗ (БВС-3) нет типа алерта «погода»: `alert_type` допускает только obstacle/zone/wind/emi. Поэтому подтип кодируем в `description`, а выбор значения `alert_type` вынесен в **открытый вопрос к заказчику**.

In [ ]:
def make_alert(row, uav_id="UAV-001"):
    state = "CONFIRMED" if row.confirmed else "WARNING"
    return {
        "alert_id": f"fog-{int(row.t)}", "uav_id": uav_id,
        "alert_type": None,                      # TODO: в ТЗ нет weather, согласовать с заказчиком
        "severity": 3 if row.confirmed else 2,
        "description": f"LENS_FOG_{state}",
        "sensor_source": "env_sensors+video",
        "confidence": round(float(max(row.p_phys, row.alarm_video)), 2),
        "timestamp": int(row.t * 1000),
        "recommended_action": "switch_to_lidar_or_thermal; mark_frames_low_confidence",
    }
first = res[res.alarm_fused].iloc[0]
make_alert(first)

## 8. Выводы и ограничения
**Что показал эксперимент (синтетика, 300 сценариев, seed=0):**
- **Видеоканал** в синтетике самый сильный: recall ≈ 1.0, ложных тревог ≈ 2 %, срабатывает около onset. Это *оптимистичная оценка*: дисперсия лапласиана падает очень круто даже при слабом размытии, а синтетическая сцена богата текстурой. На реальной однородной местности (вода, снег, ночь, дымка) резкость и так низкая, и CUSUM будет работать хуже. Это главный риск переноса результата.
- **Физический канал** даёт recall ≈ 0.74 при ложных тревогах ≈ 0.25 (включая «почти-запотевания») и медианой около −13 с. Его ценность не в точности, а в **независимости от содержимого сцены** и возможности предупредить, когда видео ещё не информативно. Он сильно зависит от неизвестных τ и offset линзы (раздел 5): без датчика температуры линзы это принципиальное ограничение.
- **Слияние по ИЛИ** наследует ложные тревоги физики. Для итогового решения разумнее двухступенчатая логика: `warn_phys` → снижение доверия к кадрам и подготовка к смене сенсора; `confirmed` (оба канала) → переключение на LiDAR/тепловизор.

**Ограничения:**
- Синтетическое размытие моделирует лишь часть эффектов реального запотевания (нет неоднородности, капель, дифракции); тепловизионный канал не моделируется.
- Пороги CUSUM и параметры физического канала подбирались на тех же генераторах; оценки оптимистичны.

**Что дальше:** проверить видеоканал на реальных кадрах с размытием/запотевшим стеклом из открытых наборов; добавить сцены с низкой текстурой (вода, снег, небо), чтобы оценить деградацию видеоканала; проверить сценарий с измеряемой температурой линзы.